Bài 3

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import math

class MinimaxGame:
    def __init__(self, size=3): # Mặc định 3x3 để Minimax không bị treo
        self.size = size
        self.board = [[' ' for _ in range(size)] for _ in range(size)]

    def check_win(self):
        for r in range(self.size):
            for c in range(self.size):
                if self.board[r][c] == ' ': continue
                for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                    count = 0
                    for i in range(3): # Điều kiện thắng 3 cho 3x3
                        nr, nc = r + dr*i, c + dc*i
                        if 0 <= nr < self.size and 0 <= nc < self.size and self.board[nr][nc] == self.board[r][c]:
                            count += 1
                        else: break
                    if count == 3: return self.board[r][c]
        if all(cell != ' ' for row in self.board for cell in row): return 'Draw'
        return None

    def minimax(self, depth, is_maximizing):
        res = self.check_win()
        if res == 'X': return 10
        if res == 'O': return -10
        if res == 'Draw' or depth == 0: return 0

        if is_maximizing:
            best_score = -math.inf
            for r in range(self.size):
                for c in range(self.size):
                    if self.board[r][c] == ' ':
                        self.board[r][c] = 'X'
                        score = self.minimax(depth - 1, False)
                        self.board[r][c] = ' '
                        best_score = max(score, best_score)
            return best_score
        else:
            best_score = math.inf
            for r in range(self.size):
                for c in range(self.size):
                    if self.board[r][c] == ' ':
                        self.board[r][c] = 'O'
                        score = self.minimax(depth - 1, True)
                        self.board[r][c] = ' '
                        best_score = min(score, best_score)
            return best_score

    def ai_move(self):
        best_score = -math.inf
        move = None
        for r in range(self.size):
            for c in range(self.size):
                if self.board[r][c] == ' ':
                    self.board[r][c] = 'X'
                    score = self.minimax(4, False) # Độ sâu giới hạn để nhanh hơn
                    self.board[r][c] = ' '
                    if score > best_score:
                        best_score = score
                        move = (r, c)
        return move

# Khởi tạo giao diện
game3 = MinimaxGame(size=3)
btns3 = []
out3 = widgets.Output()

def click3(b):
    r, c = b.pos
    if game3.board[r][c] == ' ' and not game3.check_win():
        # Người chơi đi (O)
        game3.board[r][c] = 'O'
        b.description, b.style.button_color = 'O', 'lightblue'

        # Kiểm tra sau khi người chơi đi
        winner = game3.check_win()
        if winner:
            with out3:
                clear_output()
                if winner == 'O': print("You win")
                elif winner == 'Draw': print("Draw")
            return


        move = game3.ai_move()
        if move:
            game3.board[move[0]][move[1]] = 'X'
            btns3[move[0]*game3.size + move[1]].description = 'X'
            btns3[move[0]*game3.size + move[1]].style.button_color = 'tomato'

        winner = game3.check_win()
        with out3:
            clear_output()
            if winner == 'X': print("You lose")
            elif winner == 'Draw': print("Draw")
            else: print("Your turn")

    grid3 = widgets.GridspecLayout(3, 3)
for r in range(3):
    for c in range(3):
        btn = widgets.Button(description=' ', layout=widgets.Layout(width='60px', height='60px'))
        btn.pos = (r, c)
        btn.on_click(click3)
        btns3.append(btn)
        grid3[r, c] = btn

display(grid3, out3)

GridspecLayout(children=(Button(description=' ', layout=Layout(grid_area='widget037', height='60px', width='60…

Output()

Bài 4

In [18]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import math

class AlphaBetaGame:
    def __init__(self, size=5, win_needed=4):
        self.size = size
        self.win_needed = win_needed
        self.board = [[' ' for _ in range(size)] for _ in range(size)]

    def check_win(self):
        for r in range(self.size):
            for c in range(self.size):
                if self.board[r][c] == ' ': continue
                for dr, dc in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                    count = 0
                    for i in range(self.win_needed):
                        nr, nc = r + dr*i, c + dc*i
                        if 0 <= nr < self.size and 0 <= nc < self.size and self.board[nr][nc] == self.board[r][c]:
                            count += 1
                        else: break
                    if count == self.win_needed: return self.board[r][c]
        if all(cell != ' ' for row in self.board for cell in row): return 'Draw'
        return None

    def alpha_beta(self, depth, alpha, beta, is_maximizing):
        res = self.check_win()
        if res == 'X': return 100 + depth
        if res == 'O': return -100 - depth
        if res == 'Draw' or depth == 0: return 0

        if is_maximizing:
            v = -math.inf
            for r in range(self.size):
                for c in range(self.size):
                    if self.board[r][c] == ' ':
                        self.board[r][c] = 'X'
                        v = max(v, self.alpha_beta(depth - 1, alpha, beta, False))
                        self.board[r][c] = ' '
                        alpha = max(alpha, v)
                        if beta <= alpha: break
            return v
        else:
            v = math.inf
            for r in range(self.size):
                for c in range(self.size):
                    if self.board[r][c] == ' ':
                        self.board[r][c] = 'O'
                        v = min(v, self.alpha_beta(depth - 1, alpha, beta, True))
                        self.board[r][c] = ' '
                        beta = min(beta, v)
                        if beta <= alpha: break
            return v

    def ai_move(self):
        best_v = -math.inf
        move = None
        for r in range(self.size):
            for c in range(self.size):
                if self.board[r][c] == ' ':
                    self.board[r][c] = 'X'
                    # Độ sâu 3-4 là đủ để AI 5x5 rất thông minh mà vẫn nhanh
                    v = self.alpha_beta(3, -math.inf, math.inf, False)
                    self.board[r][c] = ' '
                    if v > best_v:
                        best_v = v
                        move = (r, c)
        return move

# Khởi tạo giao diện
SIZE_4 = 5
game4 = AlphaBetaGame(size=SIZE_4, win_needed=4)
btns4 = []
out4 = widgets.Output()

def click4(b):
    r, c = b.pos
    if game4.board[r][c] == ' ' and not game4.check_win():
        game4.board[r][c] = 'O'
        b.description, b.style.button_color = 'O', 'lightblue'

        with out4:
            clear_output()
            if game4.check_win(): print("Bạn đã thắng!"); return
            print("AI đang tính toán...")

        move = game4.ai_move()
        if move:
            game4.board[move[0]][move[1]] = 'X'
            btns4[move[0]*SIZE_4 + move[1]].description = 'X'
            btns4[move[0]*SIZE_4 + move[1]].style.button_color = 'tomato'

        with out4:
            clear_output()
            res = game4.check_win()
            if res: print(f"Kết quả: {res} thắng!")

grid4 = widgets.GridspecLayout(SIZE_4, SIZE_4)
for r in range(SIZE_4):
    for c in range(SIZE_4):
        btn = widgets.Button(description=' ', layout=widgets.Layout(width='50px', height='50px'))
        btn.pos = (r, c)
        btn.on_click(click4)
        btns4.append(btn)
        grid4[r, c] = btn

display(grid4, out4)

GridspecLayout(children=(Button(description=' ', layout=Layout(grid_area='widget001', height='50px', width='50…

Output()

Bài 6

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Cấu hình game
SIZE = 5
WIN_CON = 4
game_engine = TicTacToeAI(size=SIZE, win_condition=WIN_CON)

buttons = []
output = widgets.Output()

def on_click(b):
    row, col = b.pos
    if game_engine.board[row][col] == ' ' and not game_engine.check_win(game_engine.board):
        # Người chơi đi (O)
        game_engine.board[row][col] = 'O'
        b.description = 'O'
        b.style.button_color = 'lightblue'

        winner = game_engine.check_win(game_engine.board)
        if winner:
            with output:
                clear_output()
                print(f"Kết thúc: {winner} thắng!")
            return

        # AI đi (X)
        with output:
            print("AI đang suy nghĩ...")
        move = game_engine.find_best_move(depth=2)
        if move != (-1, -1):
            game_engine.board[move[0]][move[1]] = 'X'
            btn_idx = move[0] * SIZE + move[1]
            buttons[btn_idx].description = 'X'
            buttons[btn_idx].style.button_color = 'tomato'

        winner = game_engine.check_win(game_engine.board)
        with output:
            clear_output()
            if winner: print(f"Kết thúc: {winner} thắng!")

# Tạo lưới button
grid = widgets.GridspecLayout(SIZE, SIZE)
for r in range(SIZE):
    for c in range(SIZE):
        btn = widgets.Button(description=' ', layout=widgets.Layout(width='50px', height='50px'))
        btn.pos = (r, c)
        btn.on_click(on_click)
        buttons.append(btn)
        grid[r, c] = btn

print(f"Giao diện TicTacToe {SIZE}x{SIZE} (Cần {WIN_CON} con để thắng):")
display(grid, output)